In [6]:
import yfinance as yf
import pandas as pd


raw = yf.download(
    "AAPL",
    start="2013-01-01",
    end="2024-12-31",
    auto_adjust=True,
    timeout=30,
    progress=False,
    threads=False,
)

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

print("Shape      :", raw.shape)
print("Columns    :", raw.columns.tolist())

if raw.empty:
    print("Download returned no rows. Check internet/DNS/proxy/VPN and try again.")
else:
    print("Date range :", raw.index[0], "→", raw.index[-1])
    print("Nulls      :\n", raw.isnull().sum())
    display(raw.head(3))

Shape      : (3019, 5)
Columns    : ['Close', 'High', 'Low', 'Open', 'Volume']
Date range : 2013-01-02 00:00:00 → 2024-12-30 00:00:00
Nulls      :
 Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


Price,Close,High,Low,Open,Volume
Date,,,,,
2013-01-02,16.596676,16.777144,16.372982,16.741474,560518000
2013-01-03,16.387192,16.616026,16.353940,16.561916,352965200
2013-01-04,15.930732,16.282297,15.895363,16.232116,594333600


In [8]:
#import ta
from sklearn.preprocessing import RobustScaler
import joblib, os

ModuleNotFoundError: No module named 'sklearn'

In [9]:
def validate(df):
    assert not df.isnull().values.any(), "NaNs in raw data"
    assert (df['Volume'] > 0).all(),     "Zero-volume days found"
    assert df.index.is_monotonic_increasing, "Dates out of order"
    gaps = df.index.to_series().diff().dt.days
    large_gaps = gaps[gaps > 5]          # flag holiday gaps > 5d
    if len(large_gaps):
        print(f"[warn] {len(large_gaps)} gaps > 5 days found")
    return df

In [10]:
import ta

def add_features(df):
    c, h, l, v = df['Close'], df['High'], df['Low'], df['Volume']

    df['rsi']      = ta.momentum.RSIIndicator(c, 14).rsi()
    df['macd']     = ta.trend.MACD(c).macd_diff()  # histogram, not line
    df['ema_20']   = ta.trend.EMAIndicator(c, 20).ema_indicator()
    df['ema_50']   = ta.trend.EMAIndicator(c, 50).ema_indicator()
    df['bb_width'] = (ta.volatility.BollingerBands(c).bollinger_hband() -
                      ta.volatility.BollingerBands(c).bollinger_lband()) / c
    df['obv']      = ta.volume.OnBalanceVolumeIndicator(c, v).on_balance_volume()
    df['atr']      = ta.volatility.AverageTrueRange(h, l, c).average_true_range()

    df.dropna(inplace=True)   # ~50 rows lost to indicator warmup
    return df

In [2]:
from sklearn.preprocessing import RobustScaler

def normalise(train, val, test):
    feature_cols = ['Close','rsi','macd','ema_20','ema_50',
                    'bb_width','obv','atr']

    scaler = RobustScaler()         # robust to outliers vs MinMaxScaler
    train[feature_cols] = scaler.fit_transform(train[feature_cols])
    val[feature_cols]   = scaler.transform(val[feature_cols])
    test[feature_cols]  = scaler.transform(test[feature_cols])

    import joblib
    joblib.dump(scaler, "data/processed/scaler.pkl")  # save for inference
    return train, val, test


ModuleNotFoundError: No module named 'sklearn'